# Analyse exploratoire et contrôle qualité du dataset ML

Cette analyse porte sur `data/full_sample_2000_v2`, généré avec seed 42. Elle ne lance aucun entraînement ML. Elle documente les distributions, les valeurs manquantes, la génération de `parcours_cible` et le risque de fuite de cible. La V1 est conservée séparément.

Le corpus pédagogique officiel est lu en lecture seule.

In [ ]:
from collections import Counter
import csv
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data/full_sample_2000_v2').exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'data/full_sample_2000_v2'
CORPUS_PATH = ROOT / 'donnees/corpus_pedagogique.json'
SPLITS = ('train', 'val', 'test')
rows = []
for split in SPLITS:
    with (DATA_DIR / f'{split}.csv').open(encoding='utf-8', newline='') as handle:
        rows.extend(dict(row, split=split) for row in csv.DictReader(handle))
df = pd.DataFrame(rows)
with CORPUS_PATH.open(encoding='utf-8') as handle:
    corpus = json.load(handle)
df['moyenne_scolaire'] = pd.to_numeric(df['moyenne_scolaire'])
df['competences_dict'] = df['competences'].map(json.loads)
competence_ids = [item['identifiant'] for item in corpus.get('competences', [])]
for competence_id in competence_ids:
    df[competence_id] = df['competences_dict'].map(lambda values: values.get(competence_id))
assert len(df) == 2000
df.head(3)

## 1. Cible et intégrité

In [ ]:
target_counts = df['parcours_cible'].value_counts()
target_summary = pd.DataFrame({'nombre': target_counts, 'pourcentage': (target_counts / len(df) * 100).round(2)})
print(f'Nombre de classes observées: {target_counts.size}')
print(f'Classe majoritaire: {target_counts.idxmax()} ({target_counts.max()} exemples)')
print(f'Classe minoritaire: {target_counts.idxmin()} ({target_counts.min()} exemples)')
print(f'Ratio majoritaire/minoritaire: {target_counts.max() / target_counts.min():.2f}')
target_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
target_counts.sort_values().plot.barh(ax=axes[0], color='#176b87', title='Distribution des parcours')
axes[0].set_xlabel('Nombre dexemples')
axes[1].pie(target_counts, labels=target_counts.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Part de chaque classe')
plt.tight_layout()
plt.show()

## 2. Variables et distributions

In [ ]:
print(df['moyenne_scolaire'].describe().round(3))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['moyenne_scolaire'].plot.hist(bins=20, ax=axes[0], color='#d97745', edgecolor='white', title='Distribution des moyennes')
df.boxplot(column='moyenne_scolaire', by='parcours_cible', ax=axes[1], rot=35, grid=False)
axes[1].set_title('Moyenne par cible')
plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
competence_summary = df[competence_ids].agg(['mean', 'std', 'min', 'max']).T.round(3)
competence_summary

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True)
for axis, competence_id in zip(axes.flat, competence_ids):
    df[competence_id].value_counts().sort_index().plot.bar(ax=axis, color='#2f855a', title=competence_id, rot=0)
    axis.set_xlabel('Score (0-5)')
    axis.set_ylabel('Nombre')
plt.tight_layout()
plt.show()

In [ ]:
categorical_columns = ['matieres_preferees', 'centres_interet', 'projets', 'preferences_professionnelles', 'environnement_travail']
for column in categorical_columns:
    print(f'\n{column}: {df[column].replace("", "<vide>").nunique()} modalités exactes')
    print(df[column].replace('', '<vide>').value_counts().head(10))

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for axis, column in zip(axes.flat, categorical_columns[1:]):
    values = df[column].replace('', '<vide>').value_counts().head(10).sort_values()
    values.plot.barh(ax=axis, color='#7c3f65', title=column)
plt.tight_layout()
plt.show()

## 3. Valeurs manquantes

In [ ]:
missing = df.drop(columns=['split', 'competences_dict'] + competence_ids).replace({'': pd.NA}).isna().sum()
missing_summary = pd.DataFrame({'nombre': missing, 'pourcentage': (missing / len(df) * 100).round(2)}).sort_values('nombre', ascending=False)
missing_summary

missing_summary['nombre'].plot.bar(color='#b83280', title='Valeurs manquantes par variable')
plt.ylabel('Nombre')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

`projets` est vide dans 1 181 lignes sur 2 000 (59,05 %). Cette absence est produite par le générateur (probabilité de 70 % sans projet), et non par une collecte d'étudiants réels. Elle réduit fortement la valeur de cette variable et doit être traitée explicitement avant tout entraînement.

## 4. Fuite de cible et reconstruction de la règle

In [ ]:
# Reconstruction de la règle V1, utilisée comme contrôle de fuite historique.
parcours_by_id = {item['identifiant']: item for item in corpus.get('parcours', [])}
def reconstruct_v1_target(row):
    scores = {}
    for parcours_id, parcours in parcours_by_id.items():
        score = row['moyenne_scolaire'] / 20.0
        related = parcours.get('competences', [])
        if related:
            score += sum(row[competence_id] for competence_id in related) / (len(related) * 5.0) * 0.5
        scores[parcours_id] = score
    return max(scores, key=scores.get)
df['cible_reconstruite_v1'] = df.apply(reconstruct_v1_target, axis=1)
reconstruction_rate = (df['cible_reconstruite_v1'] == df['parcours_cible']).mean()
print(f'Taux de reconstruction exacte par la règle V1: {reconstruction_rate:.1%}')
print('Parcours du corpus observés:', sorted(df['parcours_cible'].unique()))
assert reconstruction_rate < 1.0

In [ ]:
by_target_competence = df.groupby('parcours_cible')[competence_ids].mean().round(3)
by_target_mean = df.groupby('parcours_cible')['moyenne_scolaire'].agg(['mean', 'median', 'min', 'max']).round(3)
display(by_target_mean)
display(by_target_competence)
print('Lecture: chaque cible observée correspond à la compétence du parcours ayant la moyenne de score la plus élevée dans cette classe; cette relation est une conséquence de la construction de la cible.')

## 5. Verdict V2

La règle V1 ne reconstruit pas exactement les cibles V2 : le taux observé est inférieur à 100 %. La variabilité provient du bruit caché du mécanisme probabiliste. Cela ne supprime pas le caractère synthétique ni les risques de biais.

`DATASET À CORRIGER AVANT LE ML`

La V2 est méthodologiquement meilleure pour une expérimentation contrôlée, mais elle ne constitue pas une vérité sur les étudiants réels et nécessite encore une validation humaine du scénario, des distributions et de la représentativité avant tout entraînement.